# date_parser 개발/검증 노트북

담당: 이수민 (날짜 추출 / 해석 / 소비기한 판별)

이 노트북은 `date_parser` 모듈(공통 OCR 출력 `{text, confidence, bbox}`를 입력으로 받아
`year/month/day/final_date`를 출력)이 팀 규칙에 맞게 동작하는지 확인하기 위한
검증용 노트북입니다. OCR 엔진 자체는 다루지 않고, OCR 출력이라고 가정한
샘플 데이터로만 동작을 확인합니다.

실제 OCR 결과 연동은 정서현 담당 파이프라인 출력이 준비되는 대로 진행합니다.

## 1. 모듈 불러오기

프로젝트 루트에서 실행해야 `date_parser` 패키지를 찾을 수 있습니다
(`notebooks/`에서 실행 중이면 상위 폴더를 경로에 추가).

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (cwd, *cwd.parents) if (p / "date_parser").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("date_parser 패키지를 찾을 수 없습니다. 프로젝트 루트에서 실행하세요.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from date_parser import parse_expiration_date, TextBox
from date_parser.extract import extract_date_tokens
from date_parser.interpret import generate_candidates

print("date_parser 로드 완료")

## 2. 기본 동작 확인

공통 OCR 출력 형식(`text`, `confidence`, `bbox`)을 그대로 흉내 낸 예시로
`parse_expiration_date()`가 `year/month/day/final_date`를 올바르게 반환하는지 확인합니다.

In [ ]:
sample_ocr_output = [
    {"text": "소비기한", "confidence": 0.95, "bbox": [[0, 100], [60, 100], [60, 115], [0, 115]]},
    {"text": "2026.07.15", "confidence": 0.90, "bbox": [[0, 118], [90, 118], [90, 133], [0, 133]]},
]

parse_expiration_date(sample_ocr_output)

## 3. 케이스별 검증

팀 문서에서 명시된 처리 대상(YMD/DMY/MDY, 2자리 연도, 영문 월, 부분 NONE,
제조일자/소비기한 동시 존재)을 표로 정리해 한 번에 확인합니다.

In [ ]:
import pandas as pd

def make_box(text, x=0, y=0):
    return {"text": text, "confidence": 0.9, "bbox": [[x, y], [x + 80, y], [x + 80, y + 15], [x, y + 15]]}

cases = [
    ("한국어 완전 표기 (년/월/일)", [make_box("소비기한 2026년 07월 15일까지")], "2026-07-15"),
    ("4자리 연도 먼저 오는 숫자 표기", [make_box("EXP 2026.07.15")], "2026-07-15"),
    ("영문 월 이름 (일-월-연도)", [make_box("BEST BEFORE 15-JUL-2026")], "2026-07-15"),
    ("영문 월 이름 (연도-월-일)", [make_box("BBD 2026-JUL-15")], "2026-07-15"),
    ("MDY (미국식, 일/월 중 하나만 12 초과라 자동 해소)", [make_box("EXP: 07/15/2026")], "2026-07-15"),
    ("2자리 연도 + 명확한 월/일", [make_box("26.07.15")], None),
    ("부분 NONE: 일자 없음 (년/월만)", [make_box("2026년 07월 소비기한")], "2026-07-NONE"),
    ("부분 NONE: 연도 없음 (월/일만)", [make_box("07월 15일까지 섭취")], "NONE-07-15"),
    ("날짜 없음", [make_box("영양성분표 100g당")], "NONE-NONE-NONE"),
    ("부분 NONE: 달력상 불가능한 일자 (2월 30일)", [make_box("소비기한 2026.02.30")], "2026-02-NONE"),
    (
        "제조일자 vs 소비기한 동시 존재",
        [
            make_box("제조일자", y=0),
            make_box("2026.01.01", y=15),
            make_box("소비기한", y=200),
            make_box("2026.07.15", y=215),
        ],
        "2026-07-15",
    ),
]

rows = []
for label, ocr_output, expected in cases:
    result = parse_expiration_date(ocr_output)
    rows.append({"case": label, "final_date": result["final_date"], "expected": expected})

pd.DataFrame(rows)

## 4. YMD/DMY 모호성: 순서 기본값 캘리브레이션 + 제조일자 교차 검증

150장 라벨 표본(`label1.csv`)을 실제로 확인해서 기본 tie-break 순서를 다시 맞췄습니다.
처음엔 `notes`에 `DD/MM/YYYY` 태그가 21건, `MM/DD/YY` 계열이 2건 있어서 "DMY가 기본"이라고
결론 냈었는데, 이건 **선택 편향**이었습니다 — 라벨러가 태그를 남긴 건 정확히 "평소(YMD)와
다른 예외라서"였고, 태그 없는 다수는 원래 YMD라 표시할 필요가 없었던 것뿐입니다. 태그 없는
실제 사례(`26.09.24`→`2026-09-24`, `21.02.22`→`2021-02-22`)로 확인하고 YMD 우선으로
되돌렸습니다. (DMY가 MDY보다는 우선인 건 여전히 유효 — 21:2 근거.)

다만 **영문 월 이름이 있으면 반대로 DMY가 맞습니다** (`23-Jul-21`→`2021-07-23`,
`19-Mar-22`→`2022-03-19`). 영문 월 이름 자체가 "이건 숫자만 있는 한국식 표기가
아니다"라는 직접적인 텍스트 증거라서 따로 처리합니다 (`interpret.py`의
`_MONTH_NAME_ORDER_PRIOR`).

그래도 여전히 못 푸는 경우(순수 숫자, 4자리 연도 없음)가 남는데, 같은 이미지에
제조일자가 있으면 "소비기한은 제조일자보다 미래"라는 제약으로 `crossref.py`가
자동으로 해소를 시도합니다.

In [ ]:
from date_parser.crossref import apply_manufacture_constraint
from date_parser.types import DateResult

# 순수 숫자, 기본 tie-break으로는 YMD(2020-06-26)가 1순위지만 이게 틀린 경우
tokens = extract_date_tokens("20.06.26")
candidates = generate_candidates(tokens[0])

print("제조일자 정보 없이 (기본 tie-break만 적용):")
display(pd.DataFrame([
    {"rank": i + 1, "year": c.date.year, "month": c.date.month, "day": c.date.day, "score": c.score}
    for i, c in enumerate(candidates)
]))

reranked = apply_manufacture_constraint(candidates, DateResult(2024, 1, 1))
print("\n제조일자 2024-01-01이 있을 때 (교차 검증 적용 후):")
display(pd.DataFrame([
    {"rank": i + 1, "year": c.date.year, "month": c.date.month, "day": c.date.day, "score": c.score}
    for i, c in enumerate(reranked)
]))

# 영문 월 이름이 있으면 DMY가 기본값 (2자리 연도)
print("\n영문 월 이름 + 2자리 연도 (23-Jul-21):")
tokens2 = extract_date_tokens("23-Jul-21")
for c in generate_candidates(tokens2[0]):
    print(" ", c.date, c.score)

제조일자가 없으면 1순위가 `2020-06-26`(기본 tie-break, 순수 숫자라 YMD 우선)이지만,
제조일자 `2024-01-01`이 주어지면 `2020-06-26`은 "제조일자보다 과거"라 말이 안 되므로
`2026-06-20`으로 순위가 뒤집힙니다. `23-Jul-21`은 영문 월 이름이 있어서 별도 우선순위
테이블(DMY 우선)이 적용되어 `2021-07-23`이 1순위로 바로 나옵니다.

`select_final_date()`/`parse_expiration_date()` 레벨에서 실제로 어떻게 동작하는지도
확인합니다.

In [ ]:
print(parse_expiration_date([make_box("소비기한 20.06.26")]))
print(parse_expiration_date([
    make_box("제조일자 2024년 01월 01일", y=0),
    make_box("소비기한 20.06.26", y=100),
]))

이 방법은 **같은 이미지에 제조일자가 없거나, 두 해석이 모두 제조일자보다
미래라서 구분이 안 되는 경우**에는 여전히 못 풉니다 (`date_parser/README.md`
"알려진 한계" 참고). 그 나머지는 아직 미해결 상태입니다.

## 5. 다음 단계

- 정서현 OCR 파이프라인의 실제 출력 형식으로 연동 테스트
- 라벨링된 300장(`label(1).csv` + `label2.xlsx`)으로 정확도 측정
- 제조일자가 아예 없는 이미지의 YMD/DMY 모호성은 여전히 미해결 (원산지 텍스트 등
  추가 신호 필요 여부 논의)